# Benchmark Karsilastirmasi
Portfolyo olmadan: Altin, Gumus, Dolar, Euro, BIST100 ve Mevduat faizi karsilastirmasi.
Tum varliklar baslangic=100 bazinda normalize edilir.

In [7]:
# Hucre 1 - Bagimlilik kurulumu
import subprocess, sys

REQUIRED = ["yfinance", "plotly", "ipywidgets"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Bagimliliklar hazir")

Bagimliliklar hazir


In [8]:
# Hucre 2 - Importlar + Google Drive baglama
import os, sys
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive baglandi")
else:
    print("Yerel ortam - Drive mount atlandi")

Yerel ortam - Drive mount atlandi


In [ ]:
# Hucre 3 - Konfigurasyon
if IN_COLAB:
    DRIVE_BASE = "/content/drive/MyDrive/PortfolioProject/"
else:
    DRIVE_BASE = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data") + os.sep

CACHE_PATH = os.path.join(DRIVE_BASE, "cache")

# Karsilastirilacak benchmark'lar
SYMBOLS = {
    "Gram Altin": "GC=F",
    "Gram Gumus": "SI=F",
    "DOLAR":      "USDTRY=X",
    "EURO":       "EURTRY=X",
    "BIST100":    "XU100.IS",
}

TCMB_POLICY_RATE_PCT = 37  # Guncelle: mevcut TCMB politika faizi

# Varsayilan baslangic tarihi
DEFAULT_START = "2023-01-01"
DEFAULT_END   = datetime.today().strftime("%Y-%m-%d")

print(f"Config: {len(SYMBOLS)} benchmark sembol")

Config: 5 benchmark sembol


In [10]:
# Hucre 4 - lib/ import
if IN_COLAB:
    LIB_PATH = "/content/drive/MyDrive/PortfolioProject/lib"
else:
    LIB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")

if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from data_loader import fetch_prices, load_cpi_series, load_tcmb_rates
from benchmark_engine import build_benchmark_series, build_deposit_series
from chart_builder import build_performance_line_chart
from widgets import create_date_range_picker, create_currency_toggle, wire_dashboard

print("lib/ moduller yuklendi")

lib/ moduller yuklendi


In [11]:
# Hucre 5 - Veri yukle
os.makedirs(CACHE_PATH, exist_ok=True)

CPI_PATH  = os.path.join(DRIVE_BASE, "cpi_turkey.csv")
TCMB_PATH = os.path.join(DRIVE_BASE, "tcmb_rates.csv")

cpi_series  = load_cpi_series(CPI_PATH)
tcmb_rates  = load_tcmb_rates(TCMB_PATH, policy_rate_pct=TCMB_POLICY_RATE_PCT)

all_symbols = list(SYMBOLS.values()) + ["USDTRY=X"]
prices = fetch_prices(all_symbols, start=DEFAULT_START, end=DEFAULT_END, cache_path=CACHE_PATH)
fx_usdtry = prices["USDTRY=X"].dropna()

print(f"Veri hazir | Aralik: {DEFAULT_START} - {DEFAULT_END}")

Veri hazir | Aralik: 2023-01-01 - 2026-05-11


In [12]:
# Hucre 6 - Dashboard
output = widgets.Output()

CURRENCY_LABELS = {"TL": "TL (Nominal)", "USD": "USD", "REAL": "Reel (TUFE)"}
sym_to_name = {v: k for k, v in SYMBOLS.items()}

def render(start_date, end_date, currency):
    benchmark_df = build_benchmark_series(
        symbols=list(SYMBOLS.values()),
        start_date=start_date,
        end_date=end_date,
        prices=prices,
        fx_usdtry=fx_usdtry,
        cpi_series=cpi_series if currency == "REAL" else None,
        currency=currency,
    )
    deposit_series = build_deposit_series(tcmb_rates, start_date, end_date)
    benchmark_df["Mevduat"] = deposit_series

    benchmark_df = benchmark_df.rename(columns=sym_to_name)

    currency_label = CURRENCY_LABELS.get(currency, currency)

    line_fig = build_performance_line_chart(
        portfolio_series=None,
        benchmark_series=benchmark_df,
        currency_label=currency_label,
        title=f"Benchmark Karsilastirmasi ({currency_label})",
    )
    display(line_fig)

    son_degerler = benchmark_df.iloc[-1].dropna()
    getiri_df = pd.DataFrame({
        "Varlik": son_degerler.index,
        "Son Deger (baz=100)": son_degerler.values.round(2),
        "Toplam Getiri %": (son_degerler.values - 100).round(2),
    }).sort_values("Toplam Getiri %", ascending=False)

    tbl = go.Figure(go.Table(
        header=dict(
            values=["<b>Varlik</b>", "<b>Son Deger</b>", "<b>Getiri %</b>"],
            fill_color="#313244",
            font=dict(color="#cdd6f4", size=12),
            align="center",
        ),
        cells=dict(
            values=[
                getiri_df["Varlik"],
                getiri_df["Son Deger (baz=100)"],
                [f"{v:+.2f}%" for v in getiri_df["Toplam Getiri %"]],
            ],
            fill_color=[["#1e1e2e" if i % 2 == 0 else "#181825" for i in range(len(getiri_df))]],
            font=dict(color="#cdd6f4", size=11),
            align=["left", "right", "right"],
        ),
    ))
    tbl.update_layout(
        title="Donem Sonu Getiri Ozeti",
        template="plotly_dark",
        height=max(200, 38 * len(getiri_df) + 60),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    display(tbl)

start_picker, end_picker = create_date_range_picker(
    min_date=datetime(2015, 1, 1),
    max_date=datetime.today(),
    default_start=datetime.strptime(DEFAULT_START, "%Y-%m-%d"),
    default_end=datetime.today(),
)
currency_toggle = create_currency_toggle()

dashboard = wire_dashboard(
    render_fn=render,
    output_widget=output,
    start_picker=start_picker,
    end_picker=end_picker,
    currency_toggle=currency_toggle,
)

with output:
    render(DEFAULT_START, DEFAULT_END, "TL")

display(dashboard)